# Boosting Without the Brakes

**Topic:** Gradient boosting &nbsp;·&nbsp; **Difficulty:** Hard

Gradient boosting with a learning rate of 1.0 fits the training data fast and
then keeps going, memorizing noise one stump at a time. Shrink each step to 0.1
and you need many more rounds, but you get a better model.

Build the whole thing from stumps: fit the residual, take a small step, repeat.
Then track both test curves and see where each one turns around.

## The data (created in the next cell)

- `X_train` `(400, 5)` with `y_train`, and `X_test` `(400, 5)` with `y_test`.
- Squared loss, so the "gradient" you fit at each round is just the residual.
- Base learner: `DecisionTreeRegressor(max_depth=1)`, a single stump.

## Your task

With **sklearn and NumPy**, for each learning rate in `[1.0, 0.1]`:

1. Start with `F = mean(y_train)`.
2. For 200 rounds: fit a stump to the current residual `y_train - F_train`, then
   update `F <- F + lr * stump.predict(...)` on both splits. Use
   `DecisionTreeRegressor(max_depth=1, random_state=0)`.
3. Record test RMSE after every round.
4. Set `answers` to a dict with `"test_rmse_lr1"`, `"test_rmse_lr01"` (length-200
   arrays), `"best_round_lr1"`, `"best_round_lr01"` (int, 1-based round of
   minimum test RMSE), `"best_rmse_lr1"`, `"best_rmse_lr01"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the shrunk model reaches a lower minimum and takes more rounds to get
there.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    Z = r.normal(0, 1, (n, 5))
    t = (1.5 * Z[:, 0] + np.sin(2 * Z[:, 1]) - 0.8 * Z[:, 2] * Z[:, 3])
    return Z, t + r.normal(0, 0.9, n)

X_train, y_train = make(400, 61)
X_test, y_test = make(400, 62)
n_rounds = 200
print(f"train {X_train.shape}, test {X_test.shape}, {n_rounds} boosting rounds")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("test_rmse_lr1", [200], 292.0620155911695, 19.92288463284011, 1e-06),
    ("test_rmse_lr01", [200], 272.21859617080054, 18.001380206946244, 1e-06),
    ("best_round_lr1", [], 26.0, -4.353388385176978, 1e-06),
    ("best_round_lr01", [], 154.0, -25.78545428143287, 1e-06),
    ("best_rmse_lr1", [], 1.4216579178902842, -0.23803957951686344, 1e-06),
    ("best_rmse_lr01", [], 1.3107824823374172, -0.21947481669619845, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "best_rmse_lr01" in answers and "best_rmse_lr1" in answers:
    _lower = answers["best_rmse_lr01"] < answers["best_rmse_lr1"]
    _slower = answers["best_round_lr01"] > answers["best_round_lr1"]
    print(f"  {'shrinkage wins':<24} {'ok' if _lower else 'wrong'}  "
          f"({answers['best_rmse_lr01']:.4f} vs {answers['best_rmse_lr1']:.4f})")
    print(f"  {'and needs more rounds':<24} {'ok' if _slower else 'wrong'}  "
          f"(round {answers['best_round_lr01']} vs {answers['best_round_lr1']})")
    _ok = _ok and _lower and _slower

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

With `lr = 1.0` each stump's prediction is applied at full strength, so the
ensemble races to low training error and then starts fitting noise: the test
curve bottoms out early and climbs. With `lr = 0.1` each stump contributes a
tenth as much, so the model explores a much finer path through function space.
It needs many more rounds but reaches a lower minimum and sits there. This is
why `learning_rate` and `n_estimators` must always be tuned together.

```python
import numpy as np
from sklearn.tree import DecisionTreeRegressor

def boost(lr):
    F_tr = np.full(len(y_train), y_train.mean())
    F_te = np.full(len(y_test), y_train.mean())
    curve = np.zeros(n_rounds)
    for t in range(n_rounds):
        resid = y_train - F_tr
        stump = DecisionTreeRegressor(max_depth=1, random_state=0).fit(X_train, resid)
        F_tr = F_tr + lr * stump.predict(X_train)
        F_te = F_te + lr * stump.predict(X_test)
        curve[t] = np.sqrt(np.mean((F_te - y_test) ** 2))
    return curve

c1 = boost(1.0)
c01 = boost(0.1)

answers = {
    "test_rmse_lr1": c1,
    "test_rmse_lr01": c01,
    "best_round_lr1": int(np.argmin(c1) + 1),
    "best_round_lr01": int(np.argmin(c01) + 1),
    "best_rmse_lr1": float(c1.min()),
    "best_rmse_lr01": float(c01.min()),
}
```